## Grant Deduplication (S2)

Deduplicates newly-queried Dimensions grants (from S1) against tracked/historical grant sources - the last report's curated data and the GFI Grants Tracker (Airtable export) - before the survivors go to S3 for LLM scope screening.

This is a manual notebook, not an automated script: four steps pause for you to review and confirm full/partial title matches in an exported CSV before continuing. Redesigned from `SHEEP/Funding/grant_deduplication.ipynb` - see that notebook for the original prototype and its "Stella" validation notes.

Note: this notebook does not compare anything against a "ground truth" dataset. There isn't one - a new grant's scope/pillar labels come from combined LLM (S3) and human review, which is what makes them trustworthy going forward. What this notebook does is purely deduplication: telling apart grants that are already tracked (in the last report or the Grants Tracker) from genuinely new ones that need to go through S3.

**Before running:** set `RUN_TABLE` in the config cell below to the run you want to process - the name `pipeline_funding.py` printed when it asked you to run this notebook (e.g. `run_260721_1400`).

In [ ]:
import sys
from pathlib import Path
from datetime import datetime

import duckdb
import pandas as pd
import numpy as np
import pycountry
import re

sys.path.append(str(Path.cwd()))
from Funding_dedup_helpers import (
    is_empty, is_zero, normalize_title, assign_stable_row_id,
    gap_fill, gap_fill_researchers_and_orgs,
    export_highlighted_diff, export_for_review, apply_reviewed_decisions,
)

# CONFIG - edit before running
#RUN_TABLE = 'run_YYMMDD_HHMM'  # <-- set this to match the run printed by pipeline_funding.py
RUN_TABLE = 'run_260723_1526'

DB_PATH = Path('funding.db')       # self-contained in Pipeline/Funding, mirrors Publications
RAW_DATA_DIR = Path('raw_data')    # self-contained in Pipeline/Funding
REVIEW_DIR = Path('data_review')   # review CSVs (export_for_review / apply_reviewed_decisions)
AUDIT_DIR = Path('data_audit')     # highlighted-diff audit exports + final table Excel mirrors
REVIEW_DIR.mkdir(exist_ok=True)
AUDIT_DIR.mkdir(exist_ok=True)

LAST_REPORT_FILE = 'Funding2026_inscope.xlsx'
GRANTS_TRACKER_FILE = 'GrantsTracker_2026-06-30.xlsx'  # update to the latest tracker export as needed

# Whether last_report_data (LAST_REPORT_FILE) has global or Europe-only historical coverage.
# 'europe': last_report_data only has Europe grants (true through the 2026 report) - Grants
#   Tracker filtering below applies this cycle's date window to Europe grants only, and does a
#   one-time full-history backfill of non-Europe grants (never tracked before, so there's no
#   prior-cycle coverage to rely on for them).
# 'global': last_report_data already has full worldwide coverage (true from whichever cycle
#   first completes the non-Europe backfill onward) - just the incremental date window applies,
#   no region carve-out needed. Flip this to 'global' once that's confirmed true.
LAST_REPORT_SCOPE = 'europe'

# Grants Tracker date filter - only rows with EXT_Date added in this window are included.
# Defaults to the full LAST_REPORT_YEAR calendar year; edit directly for a different range.
GT_DATE_ADDED_START = '2026-01-01'
GT_DATE_ADDED_END   = '2026-12-31'

FUZZY_THRESHOLD = 85  # 0-100; rapidfuzz token_sort_ratio threshold for partial title matches

# Funding start-year filter - applied to BOTH last_report_data and grants_tracker_data (not just
# new Dimensions queries, which use their own START_YEAR/END_YEAR in S1). Rows with a missing/blank
# start-year field are EXCLUDED, not assumed in-range - see the printed NOTE lines in the load/filter
# cells below for how many rows that affects each run.
FUNDING_START_YEAR_MIN = 2020
FUNDING_START_YEAR_MAX = 2025

DEDUP_TABLE = f'{RUN_TABLE}_dedup'
RUN_DATE = datetime.today().strftime('%y%m%d')
RUN_TIMESTAMP = datetime.today().strftime('%y%m%d_%H%M')  # for audit/review filenames, so reruns don't overwrite

print(f"RUN_TABLE = {RUN_TABLE}")
print(f"DEDUP_TABLE (output, S3 reads this) = {DEDUP_TABLE}")

RUN_TABLE = run_260723_1526
DEDUP_TABLE (output, S3 reads this) = run_260723_1526_dedup


### 1. Last report's curated data

The base dataset this run gap-fills and extends. Already-tracked grants here are not re-scored - they keep whatever scope/pillar labels they were previously assigned (via LLM + human review in an earlier cycle).

In [32]:
def clean_dtypes(df):
    df = df.copy()
    datetime_cols = [
        'Date request submitted', 'Date award announced',
        'Project start date', 'Date added', 'Last modified',
    ]
    for col in datetime_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce').astype('datetime64[us]')

    year_cols = ['Year request submitted', 'Year project started', 'End date']
    for col in year_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    def _clean_months(x):
        if pd.isna(x) or isinstance(x, pd.Timestamp):
            return None
        try:
            return int(float(x))
        except (ValueError, TypeError):
            return None

    if 'Duration of award (months)' in df.columns:
        df['Duration of award (months)'] = df['Duration of award (months)'].apply(_clean_months).astype('Int64')

    if 'duration (years)' in df.columns:
        def _clean_years(x):
            if pd.isna(x) or isinstance(x, pd.Timestamp):
                return None
            try:
                val = int(float(x))
                return val if 0 <= val <= 20 else None
            except (ValueError, TypeError):
                return None
        df['duration (years)'] = df['duration (years)'].apply(_clean_years).astype('Int64')

    return df


last_report_data = pd.read_excel(RAW_DATA_DIR / LAST_REPORT_FILE)
if 'Production platform' in last_report_data.columns:
    last_report_data = last_report_data.rename(columns={'Production platform': 'AP pillar'})
last_report_data = clean_dtypes(last_report_data)
last_report_data = assign_stable_row_id(last_report_data, 'lrd_row_id')

n_loaded = len(last_report_data)

# Only grants with a funding start year in [FUNDING_START_YEAR_MIN, FUNDING_START_YEAR_MAX] are in scope.
# NOTE: rows with a missing/blank 'Year project started' are EXCLUDED here - a blank year can't be
# confirmed to fall in range, so it's dropped rather than assumed in-scope. See the printed count below.
year_filter = last_report_data['Year project started'].between(FUNDING_START_YEAR_MIN, FUNDING_START_YEAR_MAX).fillna(False)
n_missing_start_year = last_report_data['Year project started'].isna().sum()
print(f"NOTE: {n_missing_start_year} last_report_data rows have a missing/blank 'Year project started' "
      f"and are being EXCLUDED by the {FUNDING_START_YEAR_MIN}-{FUNDING_START_YEAR_MAX} start-year filter.")

last_report_data = last_report_data[year_filter].reset_index(drop=True)
last_report_data_edited = last_report_data.copy()

print(f"{n_loaded} rows loaded from '{LAST_REPORT_FILE}'")
print(f"{n_loaded - len(last_report_data)} rows dropped by the {FUNDING_START_YEAR_MIN}-{FUNDING_START_YEAR_MAX} start-year filter")
print(f"{len(last_report_data)} rows remain in last_report_data")
last_report_data.head()

1180 rows loaded from 'Funding2025_inscope.xlsx'


,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,Minority serving institution,Date added,Last modified,GFI LOS,Link to LOS,GFI partner,Tier,Year project started,End date,lrd_row_id
0,Denmark announces 1 billion kroner for plant-b...,NaN,NaN,airtable,1250000000,1250000000,DKK,177000000.0,177000000.0,162500000.0,...,NaN,2023-04-13,2023-12-04,NaN,NaN,NaN,Tier 4 (No GFI involvement),2021,<NA>,0
1,Plant2Food,The new collaborative platform Plant2Food will...,NaN,airtable,200000000,200000000,DKK,28473000.0,0.0,26000000.0,...,NaN,2023-02-27,2023-04-19,NaN,NaN,NaN,Tier 4 (No GFI involvement),2022,<NA>,1
2,CO2 as a sustainable raw material in our futur...,"In a new consortium, companies and university ...",NaN,airtable,100000000,100000000,DKK,27000000.0,0.0,13000000.0,...,NaN,2024-03-26,2025-05-02,NaN,NaN,NaN,Tier 4 (No GFI involvement),2023,<NA>,2
3,"Precision Technology: Biotechnology, smart sen...",The project has three sub-goals:\n\nDeveloping...,NaN,airtable,64200000,64200000,NOK,NaN,NaN,5585400.0,...,NaN,2023-02-28,2023-04-20,NaN,NaN,NaN,Tier 4 (No GFI involvement),2021,<NA>,3
4,SEEDFOOD: Functional and palatable plant seed ...,The Foundation has awarded one of the 2021 gra...,NaN,airtable,55900000,55900000,DKK,8172831.0,0.0,7267000.0,...,NaN,2023-04-04,2023-04-19,NaN,NaN,NaN,Tier 4 (No GFI involvement),2021,<NA>,4


### 2. GFI Grants Tracker data (Airtable export)

In [33]:
grants_tracker_data = pd.read_excel(RAW_DATA_DIR / GRANTS_TRACKER_FILE)
grants_tracker_data = assign_stable_row_id(grants_tracker_data, 'gt_row_id')  # stamped before filtering, reflects the raw export's row position

grants_tracker_data['EXT_Date added'] = pd.to_datetime(grants_tracker_data['EXT_Date added'], errors='coerce')
grants_tracker_data['EXT_Years project starts'] = pd.to_numeric(grants_tracker_data['EXT_Years project starts'], errors='coerce')

date_filter = (
    (grants_tracker_data['EXT_Date added'] >= GT_DATE_ADDED_START) &
    (grants_tracker_data['EXT_Date added'] <= GT_DATE_ADDED_END)
)

# Only grants with a funding start year in [FUNDING_START_YEAR_MIN, FUNDING_START_YEAR_MAX] are in scope.
# NOTE: rows with a missing/blank 'EXT_Years project starts' are EXCLUDED here - a blank year can't
# be confirmed to fall in range, so it's dropped rather than assumed in-scope. See the printed count below.
year_filter = grants_tracker_data['EXT_Years project starts'].between(FUNDING_START_YEAR_MIN, FUNDING_START_YEAR_MAX).fillna(False)
n_missing_start_year = grants_tracker_data['EXT_Years project starts'].isna().sum()
print(f"NOTE: {n_missing_start_year} grants_tracker rows have a missing/blank 'EXT_Years project starts' "
      f"and are being EXCLUDED by the {FUNDING_START_YEAR_MIN}-{FUNDING_START_YEAR_MAX} start-year filter.")

if LAST_REPORT_SCOPE == 'europe':
    europe_filter = (
        (grants_tracker_data['EXT_Funder region'] == 'Europe') |
        (grants_tracker_data['EXT_PI organization region'] == 'Europe')
    )
    # Europe: incremental slice only (already covered historically). Non-Europe: full backlog,
    # any date, one-time catch-up. Blank/missing region ends up on the non-Europe side (~europe_filter
    # is True whenever both region columns are NaN, since `NaN == 'Europe'` evaluates False) -
    # included in the backfill per your call above.
    combined_filter = (europe_filter & date_filter) | ~europe_filter
elif LAST_REPORT_SCOPE == 'global':
    # last_report_data already has full worldwide coverage - just the incremental window applies.
    combined_filter = date_filter
else:
    raise ValueError(f"LAST_REPORT_SCOPE must be 'europe' or 'global', got {LAST_REPORT_SCOPE!r}")

combined_filter = combined_filter & year_filter  # start-year requirement applies regardless of LAST_REPORT_SCOPE

grants_tracker_data = grants_tracker_data[combined_filter].reset_index(drop=True)
print(f"{len(grants_tracker_data)} grants included after filtering")

# Rename Gov't -> Gov throughout the pipeline to avoid apostrophe quoting issues
grants_tracker_data = grants_tracker_data.rename(columns={
    "INT_Gov't contribution (actual currency)": 'INT_Gov contribution (actual currency)',
    "EXT_Gov't contribution (USD)":             'EXT_Gov contribution (USD)',
})

grants_tracker_data_edited = grants_tracker_data.copy()
grants_tracker_data.head()

550 grants included after filtering


,EXT_Title,INT_Total amount (actual currency),INT_Gov contribution (actual currency),INT_Currency type,EXT_Total amount (USD),EXT_Gov contribution (USD),EXT_Funding decision,EXT_URL for announcement,EXT_Notes (external),INT_Notes INTERNAL ONLY,...,INT_End Date (Formula),INT_Funding call,INT_Success rate,INT_Minority serving institution?,EXT_Abstract,Dimensions.ai grant ID,Program type,Research area,Flags,gt_row_id
0,Systems Biology of Hydrogen Oxidising Bacteria...,0,0,GBP,0,0.0,Awarded,https://gtr.ukri.org/projects?ref=studentship-...,NaN,NaN,...,ERROR,NaN,NaN,NaN,The world's population is predicted to reach 1...,grant.9452322,NaN,PF,NaN,1250
1,Food processing residues to climate smart food...,800000,800000,SEK,76376,76376.0,NaN,https://www.vr.se/swecris.html#/project/2022-0...,NaN,NaN,...,ERROR,NaN,NaN,NaN,NaN,grant.4456273,NaN,BF,NaN,1251
2,MET2FOOD,108738,108738,GBP,139436,139436.0,NaN,https://gtr.ukri.org/projects?ref=91600,NaN,NaN,...,ERROR,NaN,NaN,NaN,NaN,grant.9555967,NaN,BF,NaN,1252
3,Understanding the functional properties of mic...,0,0,GBP,0,0.0,Awarded,https://gtr.ukri.org/projects?ref=studentship-...,NaN,NaN,...,ERROR,NaN,NaN,NaN,Abstract\nThere is growing demand for sustaina...,grant.13022069,NaN,BF,NaN,1253
4,Machine-learning generated nucleases for accel...,50000,50000,GBP,61439,61439.0,NaN,https://gtr.ukri.org/projects?ref=10072768,NaN,NaN,...,ERROR,NaN,NaN,NaN,NaN,grant.13883411,NaN,PF,NaN,1254


In [34]:
valid_countries = set()
for c in pycountry.countries:
    valid_countries.add(c.name.strip().lower())
    if hasattr(c, 'common_name'):
        valid_countries.add(c.common_name.strip().lower())

valid_countries.update({'czech republic', 'russia', 'turkey', 'uk'})

def _is_valid_country_cell(val):
    if pd.isna(val) or str(val).strip() == '':
        return False
    first_part = re.split(r'[;,]', str(val))[0].strip().lower()
    return first_part in valid_countries

before_count = grants_tracker_data['EXT_PI organization country'].notna().sum()
removed_vals = (
    grants_tracker_data['EXT_PI organization country']
    .dropna()
    .loc[lambda s: ~s.apply(_is_valid_country_cell)]
    .unique()
)

for _df in (grants_tracker_data, grants_tracker_data_edited):
    _df['EXT_PI organization country'] = _df['EXT_PI organization country'].apply(
        lambda val: val if _is_valid_country_cell(val) else None
    )

after_count = grants_tracker_data['EXT_PI organization country'].notna().sum()
print(f"Nulled out {before_count - after_count} non-country values ({before_count} -> {after_count} filled)")
print(f"Values removed: {sorted(removed_vals)}")

Nulled out 29 non-country values (467 -> 438 filled)
Values removed: ['Alain Baranger', 'Andre Rastica', 'Andrew Clayton', 'Andrew Stacey', 'Aurélien Ducrey, Aurélien Ducrey', 'Christer Heimtoft', 'D.K. Karefyllakis', 'Diana Maria Condeço Marques', 'Diego Moretti, Laila Hammer, Hilaj Nikolin, Pornpimol Scheuchzer', 'Ecevit Yilmaz', 'Eric Öste', 'Ernst Langthaler', 'Fabian Pfrengle', 'Fengzheng Gao, Fengzheng Gao', 'Filipa Soares', 'Gunnar Backman', 'Harriet Gregory', 'Karen Fairlie-Clarke', 'Karima Karagussova', 'Kevin STEPHENS', 'Ky Son Chu', 'Leif Horsfelt Skibsted', 'Sarah Gaunt', 'Steve Skill', 'Stig A. Borgvang', 'Thomas Brunner, Bao Duong Pham, Mathilde Delley, Barbara Franco Lucas, Franziska Götze, Isabel Häberlil, Reto Huwiler, Evelyn Markoni', 'Veronika Temml', 'Véronique Cheynier']


### 3. Dimensions data (from S1's `{RUN_TABLE}`)

Reads S1's automated Dimensions grants query output directly from `funding.db`, replacing the 4 manually-exported Dimensions Excel files the original prototype used.

**Adapter cell below**: normalizes S1's DSL-native column names/shapes into the flat column names the rest of this notebook expects (matching the legacy manual-export format it was originally built around). If S1's field shapes change, only this cell should need updating - flagged in the implementation plan as needing live validation against a real `dsl.query()` call.

Also handles the case where `RUN_TABLE` already holds data in the legacy manual-export schema (e.g. a test table built directly from the raw `Dimensions-Grant-*.xlsx` files, which already has columns like `Grant ID`/`Researchers` rather than S1's lowercase `id`/`researchers`) - detected automatically, no adapting applied in that case since it's already in the target shape.

In [35]:
import ast

con = duckdb.connect(str(DB_PATH))
dimensions_raw = con.sql(f"SELECT * FROM {RUN_TABLE}").df()
con.close()
print(f"{len(dimensions_raw)} rows loaded from '{RUN_TABLE}'")

def _maybe_parse_literal(val):
    """Some Dimensions DSL fields with deeply nested structure (e.g. researchers, whose
    dicts contain a further nested research_orgs list) come back from DuckDB as a Python-repr
    string rather than a real list/dict - detect and parse that case back into real objects."""
    if isinstance(val, str) and val.strip().startswith(('[', '{')):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return val
    return val


def _join_list(val):
    """Flatten a dimcli-style nested field (list of dicts, list of strings, or a plain
    scalar) into a semicolon-joined string, matching the legacy manual-export format."""
    val = _maybe_parse_literal(val)
    if val is None or (not isinstance(val, (list, tuple, np.ndarray)) and pd.isna(val)):
        return None
    if isinstance(val, str):
        return val
    if not isinstance(val, (list, tuple, np.ndarray)):
        return str(val)
    parts = []
    for item in val:
        if isinstance(item, dict):
            name = item.get('name')
            if name is None and ('first_name' in item or 'last_name' in item):
                name = f"{item.get('first_name', '')} {item.get('last_name', '')}".strip()
            parts.append(str(name if name is not None else item))
        else:
            parts.append(str(item))
    parts = [p for p in parts if p]
    return '; '.join(parts) if parts else None


def _first_or_none(val):
    joined = _join_list(val)
    if not joined:
        return None
    return joined.split(';')[0].strip()


def _derive_researchers_from_investigators(val):
    """Build a PI-first, semicolon-joined name list from Dimensions' investigators field
    (which has an explicit 'role' per person: 'PI'/'Co-PI'/etc.), rather than researchers
    (a flat, role-less contributor list that doesn't reliably put the actual PI first - or
    even include them at all in some cases)."""
    val = _maybe_parse_literal(val)
    if val is None or (not isinstance(val, (list, tuple, np.ndarray)) and pd.isna(val)):
        return None
    if not isinstance(val, (list, tuple, np.ndarray)):
        return None
    pis, others = [], []
    for item in val:
        if not isinstance(item, dict):
            continue
        name = f"{item.get('first_name') or ''} {item.get('last_name') or ''}".strip()
        if not name:
            continue
        (pis if item.get('role') == 'PI' else others).append(name)
    ordered = pis + others
    return '; '.join(ordered) if ordered else None


# Dimensions exposes 8 currency-converted funding amounts (no generic native-currency amount -
# that field was deprecated/removed from the grants DSL source in 2018). Where funding_currency
# matches one of these 8, we can back out a genuine native-currency figure; otherwise there's no
# way to get one via the API at all.
_CURRENCY_TO_FIELD = {
    'USD': 'funding_usd', 'EUR': 'funding_eur', 'GBP': 'funding_gbp', 'AUD': 'funding_aud',
    'CAD': 'funding_cad', 'CHF': 'funding_chf', 'JPY': 'funding_jpy', 'NZD': 'funding_nzd',
}

def _native_currency_amount(row):
    currency = row.get('funding_currency')
    if currency is None or pd.isna(currency):
        return None
    field = _CURRENCY_TO_FIELD.get(str(currency).strip().upper())
    return row.get(field) if field else None


if 'id' in dimensions_raw.columns:
    # S1's DSL-schema output - needs adapting into the legacy flat column shape
    print("Detected S1 (DSL) schema - applying adapter.")
    dimensions_data = pd.DataFrame({
        'Grant ID': dimensions_raw['id'],
        'Title translated': dimensions_raw.get('title'),
        'Title': dimensions_raw.get('original_title'),
        'Abstract translated': dimensions_raw.get('abstract'),
        'Abstract': None,  # not queried by S1 - grants DSL has no separate original-language abstract field
        'Researchers': dimensions_raw['investigators'].apply(_derive_researchers_from_investigators) if 'investigators' in dimensions_raw else None,
        'Research Organization - standardized': dimensions_raw['research_org_names'].apply(_join_list) if 'research_org_names' in dimensions_raw else None,
        # Currency/amount columns: 'Total amount' is the grant's amount in its own native
        # currency, backed out via _native_currency_amount above where funding_currency matches
        # one of the 8 Dimensions exposes - None otherwise (no API field can give us this for e.g.
        # NOK/SEK-funded grants). 'Total amount (USD)'/'(EUR)' are Dimensions' direct conversions.
        'Currency': dimensions_raw.get('funding_currency'),
        'Total amount': dimensions_raw.apply(_native_currency_amount, axis=1) if len(dimensions_raw) else None,
        'Total amount (USD)': dimensions_raw.get('funding_usd'),
        'Total amount (EUR)': dimensions_raw.get('funding_eur'),
        'Start date': dimensions_raw.get('start_date'),
        'Start Year': dimensions_raw.get('start_year'),
        'End Year': pd.to_datetime(dimensions_raw['end_date'], errors='coerce').dt.year if 'end_date' in dimensions_raw else None,
        'State of standardized research organization': None,  # not queried by S1
        'Country of standardized research organization': dimensions_raw['research_org_countries'].apply(_join_list) if 'research_org_countries' in dimensions_raw else None,
        'Funder': dimensions_raw['funder_org_name'].apply(_join_list) if 'funder_org_name' in dimensions_raw else None,
        'Funder Country': dimensions_raw['funder_org_countries'].apply(_join_list) if 'funder_org_countries' in dimensions_raw else None,
        'Source Linkout': dimensions_raw['linkout'].apply(_first_or_none) if 'linkout' in dimensions_raw else None,
    })
elif 'Grant ID' in dimensions_raw.columns:
    # Already in the legacy manual-export schema (e.g. a test table loaded straight from the
    # raw Dimensions-Grant-*.xlsx files) - nothing to adapt.
    print("Detected legacy manual-export schema - using as-is, no adapting needed.")
    dimensions_data = dimensions_raw.copy()
else:
    raise ValueError(
        f"'{RUN_TABLE}' has neither an 'id' column (S1/DSL schema) nor a 'Grant ID' column "
        f"(legacy manual-export schema) - can't tell which format this table is in."
    )

# Defensive dedup - S1 already dedupes by id, but cheap to re-check across the S1/S2 boundary
before = len(dimensions_data)
dimensions_data = dimensions_data.drop_duplicates(subset='Grant ID').reset_index(drop=True)
print(f"Removed {before - len(dimensions_data)} duplicates ({before} -> {len(dimensions_data)} rows)")

dimensions_data.head()

36 rows loaded from 'run_260723_1526'
Detected S1 (DSL) schema - applying adapter.
Removed 0 duplicates (36 -> 36 rows)


,Grant ID,Title translated,Title,Abstract translated,Abstract,Researchers,Research Organization - standardized,Currency,Total amount,Total amount (USD),Total amount (EUR),Start date,Start Year,End Year,State of standardized research organization,Country of standardized research organization,Funder,Funder Country,Source Linkout
0,grant.15280337,Enrichment of zein from corn gluten and produc...,Anreichern von Zein aus Maiskleber und Herstel...,NaN,None,NaN,Research Association of the German Food Industry,EUR,152526.0,177877.0,152526.0,2025-12-01,2025,2027.0,None,Germany,Federal Ministry for Economic Affairs and Clim...,Germany,https://foerderportal.bund.de/foekat/jsp/Suche...
1,grant.15218957,Evaluation of the Nursery Milk Scheme in England,Evaluation of the Nursery Milk Scheme in England,Our project will evaluate the nursery milk sch...,None,Joanne Pearce; Claire Wall; Jordan Beaumont; P...,Sheffield Hallam University,GBP,614350.0,825933.0,708458.0,2025-11-01,2025,2028.0,None,United Kingdom,National Institute for Health and Care Research,United Kingdom,https://fundingawards.nihr.ac.uk/award/NIHR208579
2,grant.15060106,An Exclusive Sustainability Transition of Fash...,"""Een Exclusieve Duurzaamheidstransitie van de ...",This project takes the ecologically destructiv...,None,Giselinde Kuipers,KU Leuven,NaN,NaN,NaN,NaN,2025-10-01,2025,NaN,None,Belgium,Research Foundation - Flanders,Belgium,https://researchportal.be/nl/project/een-exclu...
3,grant.15149727,Masking Agents To Promote Ingestion Of Organic...,Masking Agents To Promote Ingestion Of Organic...,This STTR Phase I project is focused on novel ...,None,NaN,Foresight Science & Technology (United States),USD,181500.0,181500.0,156933.0,2025-09-15,2025,2026.0,None,United States,National Institute of Food and Agriculture,United States,https://portal.nifa.usda.gov/web/crisprojectpa...
4,grant.15082917,Identification of Biomarkers of Plant-Rich Die...,Identification of Biomarkers of Plant-Rich Die...,Abstract: One potential mechanism through whic...,None,FRANK B HU; QIBIN QI; SHENGMIN SANG,Harvard University,USD,733785.0,733785.0,649963.0,2025-09-15,2025,2030.0,None,United States,National Heart Lung and Blood Institute,United States,https://reporter.nih.gov/project-details/11224390


#### Dimensions -> last report's data, exact ID match

Gap-fills the last report's data from Dimensions (never overwrites non-empty cells, except funding columns where an existing 0 is treated as fillable), then removes matched rows from `dimensions_data` - no manual review needed, this is a real business-key match.

In [36]:
col_map = {
    'Title translated':                              'Title',
    'Title':                                         'Original title',
    'Abstract translated':                           'Abstract',
    'Total amount':                                  'Total amount',
    'Currency':                                      'Currency',
    'Total amount (USD)':                            'Total amount (USD)',
    'Total amount (EUR)':                            'Total amount (EUR)',
    'Start date':                                    'Project start date',
    'Start Year':                                    'Year project started',
    'End Year':                                      'End date',
    'State of standardized research organization':   'PI organisation state',
    'Country of standardized research organization': 'PI organisation country',
    'Funder':                                        'Funder name',
    'Funder Country':                                'Funder Country',
    'Source Linkout':                                'URL for announcement',
}
funding_cols_lrd = {'Total amount', 'Total amount (USD)', 'Total amount (EUR)'}

lrd_id_map = {}
for idx, val in last_report_data['Identification code'].items():
    if not is_empty(val):
        lrd_id_map.setdefault(str(val).strip(), []).append(idx)

matched_ids = set()
changed_indices = set()
cells_filled = 0

for _, dim_row in dimensions_data.iterrows():
    grant_id = str(dim_row['Grant ID']).strip()
    if grant_id not in lrd_id_map:
        continue
    matched_ids.add(grant_id)
    for lrd_idx in lrd_id_map[grant_id]:
        cells_filled += gap_fill(dim_row, last_report_data_edited, lrd_idx, col_map, funding_cols_lrd, changed_indices)
        cells_filled += gap_fill_researchers_and_orgs(
            dim_row, last_report_data_edited, lrd_idx,
            pi_col='Project lead (PI)', collab_col='Collaborator names',
            org_pi_col='PI organisation', org_collab_col='Collaborator institutions',
            changed_indices=changed_indices,
        )

before = len(dimensions_data)
dimensions_data = dimensions_data[
    ~dimensions_data['Grant ID'].astype(str).str.strip().isin(matched_ids)
].reset_index(drop=True)
after = len(dimensions_data)

print(f"Matched {len(matched_ids)} grants with last_report_data")
print(f"Filled {cells_filled} missing values across {len(changed_indices)} rows")
print(f"Removed {before - after} rows from dimensions_data ({after} remaining)")

Matched 0 grants with last_report_data
Filled 0 missing values across 0 rows
Removed 0 rows from dimensions_data (36 remaining)


In [37]:
export_highlighted_diff(last_report_data, last_report_data_edited, changed_indices,
                         AUDIT_DIR / f'{RUN_TIMESTAMP}_dim_ID_matches_last_report_data.xlsx')

,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,Minority serving institution,Date added,Last modified,GFI LOS,Link to LOS,GFI partner,Tier,Year project started,End date,lrd_row_id


#### Dimensions -> grants tracker, exact ID match

In [38]:
gt_col_map = {
    'Title translated':                              'EXT_Title',
    'Abstract translated':                           'EXT_Abstract',
    'Total amount':                                  'INT_Total amount (actual currency)',
    'Currency':                                      'INT_Currency type',
    'Total amount (USD)':                            'EXT_Total amount (USD)',
    'Start date':                                    'EXT_Project start date (estimated)',
    'End Year':                                      'INT_End date',
    'State of standardized research organization':   'EXT_PI organisation state',
    'Country of standardized research organization': 'EXT_PI organization country',
    'Funder':                                        'EXT_Funder name',
    'Funder Country':                                'EXT_Funder country',
    'Source Linkout':                                'EXT_URL for announcement',
}
funding_cols_gt = {'INT_Total amount (actual currency)', 'EXT_Total amount (USD)'}

gt_id_map = {}
for idx, val in grants_tracker_data['Dimensions.ai grant ID'].items():
    if not is_empty(val):
        gt_id_map.setdefault(str(val).strip(), []).append(idx)

gt_matched_ids = set()
gt_changed_indices = set()
gt_cells_filled = 0

for _, dim_row in dimensions_data.iterrows():
    grant_id = str(dim_row['Grant ID']).strip()
    if grant_id not in gt_id_map:
        continue
    gt_matched_ids.add(grant_id)
    for gt_idx in gt_id_map[grant_id]:
        gt_cells_filled += gap_fill(dim_row, grants_tracker_data_edited, gt_idx, gt_col_map, funding_cols_gt, gt_changed_indices)
        gt_cells_filled += gap_fill_researchers_and_orgs(
            dim_row, grants_tracker_data_edited, gt_idx,
            pi_col='EXT_Project lead (PI)', collab_col='EXT_Collaborator names',
            org_pi_col='EXT_PI organization', org_collab_col='EXT_Collaborator organizations',
            changed_indices=gt_changed_indices,
        )

before = len(dimensions_data)
dimensions_data = dimensions_data[
    ~dimensions_data['Grant ID'].astype(str).str.strip().isin(gt_matched_ids)
].reset_index(drop=True)
after = len(dimensions_data)

print(f"Matched {len(gt_matched_ids)} grants with grants_tracker_data")
print(f"Filled {gt_cells_filled} missing values across {len(gt_changed_indices)} rows")
print(f"Removed {before - after} rows from dimensions_data ({after} remaining)")

Matched 4 grants with grants_tracker_data
Filled 2 missing values across 1 rows
Removed 4 rows from dimensions_data (32 remaining)


In [39]:
export_highlighted_diff(grants_tracker_data, grants_tracker_data_edited, gt_changed_indices,
                         AUDIT_DIR / f'{RUN_TIMESTAMP}_dim_ID_matches_gt_data.xlsx')

,EXT_Title,INT_Total amount (actual currency),INT_Gov contribution (actual currency),INT_Currency type,EXT_Total amount (USD),EXT_Gov contribution (USD),EXT_Funding decision,EXT_URL for announcement,EXT_Notes (external),INT_Notes INTERNAL ONLY,...,INT_End Date (Formula),INT_Funding call,INT_Success rate,INT_Minority serving institution?,EXT_Abstract,Dimensions.ai grant ID,Program type,Research area,Flags,gt_row_id
461,EUROPEAN MICROALGAE ALLIANCE (ALLIANCE):\n IN...,7360662,NaN,EUR,7811893,NaN,Awarded,https://cordis.europa.eu/project/id/101214199,NaN,NaN,...,ERROR,NaN,NaN,NaN,ALLIANCE aims to broaden the uptake of microal...,grant.14955120,NaN,NaN,NaN,1787


### 4. Grants Tracker vs last report's data
#### a. Dimensions ID match

In [40]:
last_report_data_pre_gt = last_report_data_edited.copy()

gt_lrd_col_map = {
    'EXT_Title':                              'Title',
    'EXT_Abstract':                           'Abstract',
    'INT_Total amount (actual currency)':     'Total amount',
    'INT_Gov contribution (actual currency)': 'Gov contribution',
    'INT_Currency type':                      'Currency',
    'EXT_Total amount (USD)':                 'Total amount (USD)',
    'EXT_Gov contribution (USD)':             'Gov contribution (USD)',
    'EXT_Funding decision':                   'Funding decision',
    'EXT_Project start date (estimated)':     'Project start date',
    'INT_End date':                           'End date',
    'EXT_Project lead (PI)':                  'Project lead (PI)',
    'EXT_PI organization':                    'PI organisation',
    'EXT_PI organization type':               'PI organisation type',
    'EXT_PI organization country':            'PI organisation country',
    'EXT_PI organization region':              'PI organisation region',
    'PI organisation state':                  'PI organisation state',
    'EXT_Collaborator names':                 'Collaborator names',
    'EXT_Collaborator organizations':         'Collaborator institutions',
    'EXT_Funder name':                        'Funder name',
    'EXT_Funder type':                        'Funder type',
    'EXT_Funder country':                     'Funder Country',
    'EXT_Funder region':                      'Funder region',
    'EXT_Production platform':                'AP pillar',
    'EXT_End product type':                   'End product type',
    'EXT_Award purpose':                      'Award purpose',
    'EXT_URL for announcement':               'URL for announcement',
    'EXT_Date added':                         'Date added',
    'EXT_Last modified':                      'Last modified',
    'EXT_Years project starts':               'Year project started',
    'INT_GFI grantee?':                       'GFI grantee',
    'INT_GFI Los?':                            'GFI LOS',
    'INT_Link to Los':                        'Link to LOS',
    'INT_GFI partner?':                       'GFI partner',
    'INT_Tier':                                'Tier',
}
funding_cols_lrd2 = {'Total amount', 'Total amount (USD)', 'Gov contribution', 'Gov contribution (USD)'}

lrd2_id_map = {}
for idx, val in last_report_data_edited['Identification code'].items():
    if not is_empty(val):
        lrd2_id_map.setdefault(str(val).strip(), []).append(idx)

gt_lrd_matched_ids = set()
gt_lrd_changed_indices = set()
gt_lrd_cells_filled = 0

for _, gt_row in grants_tracker_data_edited.iterrows():
    grant_id = str(gt_row['Dimensions.ai grant ID']).strip()
    if grant_id not in lrd2_id_map:
        continue
    gt_lrd_matched_ids.add(grant_id)
    for lrd_idx in lrd2_id_map[grant_id]:
        gt_lrd_cells_filled += gap_fill(gt_row, last_report_data_edited, lrd_idx, gt_lrd_col_map, funding_cols_lrd2, gt_lrd_changed_indices)

before = len(grants_tracker_data_edited)
grants_tracker_data_edited = grants_tracker_data_edited[
    ~grants_tracker_data_edited['Dimensions.ai grant ID'].astype(str).str.strip().isin(gt_lrd_matched_ids)
].reset_index(drop=True)
after = len(grants_tracker_data_edited)

print(f"Matched {len(gt_lrd_matched_ids)} grants between grants_tracker_data_edited and last_report_data_edited")
print(f"Filled {gt_lrd_cells_filled} missing values across {len(gt_lrd_changed_indices)} rows")
print(f"Removed {before - after} rows from grants_tracker_data_edited ({after} remaining)")

Matched 182 grants between grants_tracker_data_edited and last_report_data_edited
Filled 497 missing values across 149 rows
Removed 187 rows from grants_tracker_data_edited (363 remaining)


In [41]:
# OPTIONAL - shows which grants were duplicated within the grants tracker data
dupes = grants_tracker_data[
    grants_tracker_data['Dimensions.ai grant ID'].isin(gt_lrd_matched_ids)
    & grants_tracker_data['Dimensions.ai grant ID'].duplicated(keep=False)
][['Dimensions.ai grant ID', 'EXT_Title']].sort_values('Dimensions.ai grant ID')
dupes

,Dimensions.ai grant ID,EXT_Title
110,grant.12941157,CIRCular valorisation of industrial ALGAE wast...
393,grant.12941157,CIRCular\n valorisation of industrial ALGAE w...
115,grant.13253023,EIT Food Activities
440,grant.13253023,EIT Food Activities (TASTE2MEAT)
5,grant.13879477,Fermentation optimisation for a palm oil alter...
372,grant.13879477,Fermentation optimisation for a palm oil alter...
218,grant.13909296,Harnessing genetic diversity of the novel rape...
405,grant.13909296,Harnessing genetic diversity of the novel\n r...
78,grant.9965207,Novel texturized hybrid foods targeting future...
370,grant.9965207,Novel texturized\n hybrid foods targeting fut...


In [42]:
export_highlighted_diff(last_report_data_pre_gt, last_report_data_edited, gt_lrd_changed_indices,
                         AUDIT_DIR / f'{RUN_TIMESTAMP}_gt_changes_last_report_data.xlsx')

,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,Minority serving institution,Date added,Last modified,GFI LOS,Link to LOS,GFI partner,Tier,Year project started,End date,lrd_row_id
8,Green technology for plant-based food (GreenPl...,To maintain national food self-sufficiency and...,NaN,airtable,27600000,27600000,NOK,2604048.0,2604048.0,2401200.00,...,NaN,2023-04-21,2023-04-21,NaN,NaN,NaN,Tier 4 (No GFI involvement),2021,<NA>,8
60,"ECOnti - Accelerated, low ecological footprint...",Initial situation: \nMicrobial processes are e...,NaN,airtable,3600000,2700000,EUR,0.0,0.0,3600000.00,...,NaN,2025-01-14,2025-05-02,NaN,NaN,NaN,NaN,2023,<NA>,60
135,Extruded and 3D printed vegan support structur...,No abstract,NaN,airtable,768825,768825,EUR,824278.0,824278.0,768825.00,...,NaN,2025-01-14,2025-05-02,NaN,NaN,NaN,NaN,2022,<NA>,135
138,SUSTAINER: Sustainable production of structura...,Biomaterials are materials engineered to inter...,NaN,airtable,750000,750000,EUR,0.0,0.0,750000.00,...,NaN,2025-01-14,2025-05-02,NaN,NaN,NaN,NaN,2023,<NA>,138
163,Surface-active algae protein isolates for vega...,No abstract,NaN,airtable,521407,521407,EUR,563410.0,563410.0,521407.00,...,NaN,2025-01-14,2025-05-02,NaN,NaN,NaN,NaN,2023,<NA>,163
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
933,A novel support material for 3D bioprinting an...,"""Three-dimensional (3D) bioprinting holds grea...",A novel support material for 3D bioprinting an...,Dimensions,150000,150000,EUR,160500.0,160371.0,150000.00,...,NaN,2025-01-14,2026-03-17,"""Three-dimensional (3D) bioprinting holds grea...",NaN,NaN,NaN,2022,2023,933
934,Whole-cut cell-based fish fillet production by...,NaN,Whole-cut cell-based fish fillet production by...,Dimensions,20590,20590,EUR,22221.0,22221.0,NaN,...,NaN,2025-01-14,2026-03-17,NaN,NaN,NaN,NaN,2022,2026,934
936,Advanced Cellular Hierarchical Tissue-Imitatio...,ACHIEVE focuses on the application of Excluded...,Advanced Cellular Hierarchical Tissue-Imitatio...,Dimensions,2076770,2076770,EUR,2233939.0,2246117.0,2076770.00,...,NaN,2025-01-14,2026-03-17,ACHIEVE focuses on the application of Excluded...,NaN,NaN,NaN,2021,2026,936
938,3D Cell Scaffolds for Clean Meat Cultivation,Conventional farming of animal-based protein h...,3D Cell Scaffolds for Clean Meat Cultivation,Dimensions,230616,230616,CHF,260104.0,168076.0,246759.12,...,NaN,2025-01-14,2026-03-17,Conventional farming of animal-based protein h...,NaN,NaN,NaN,2023,2024,938


#### b. Title-based match - REVIEW POINT 1 (exact title)

In [43]:
lrd_title_map = {}
for idx, val in last_report_data_edited['Title'].items():
    norm = normalize_title(val)
    if norm:
        lrd_title_map.setdefault(norm, []).append(idx)

title_match_rows = []
for gt_idx, gt_row in grants_tracker_data_edited.iterrows():
    norm = normalize_title(gt_row.get('EXT_Title'))
    if norm and norm in lrd_title_map:
        for lrd_idx in lrd_title_map[norm]:
            title_match_rows.append({
                'gt_row_id':               gt_row['gt_row_id'],
                'lrd_row_id':              last_report_data_edited.at[lrd_idx, 'lrd_row_id'],
                'GT index':                gt_idx,
                'LRD index':               lrd_idx,
                'Title':                   gt_row.get('EXT_Title'),
                'GT Dimensions ID':        gt_row.get('Dimensions.ai grant ID'),
                'LRD Identification code': last_report_data_edited.at[lrd_idx, 'Identification code'],
                'GT Funder':               gt_row.get('EXT_Funder name'),
                'LRD Funder':              last_report_data_edited.at[lrd_idx, 'Funder name'],
                'GT Total (USD)':          gt_row.get('EXT_Total amount (USD)'),
                'LRD Total (USD)':         last_report_data_edited.at[lrd_idx, 'Total amount (USD)'],
                'GT Project start date':   gt_row.get('EXT_Project start date (estimated)'),
                'LRD Project start date':  last_report_data_edited.at[lrd_idx, 'Project start date'],
            })

title_matches_df = pd.DataFrame(title_match_rows, columns=[
    'gt_row_id', 'lrd_row_id', 'GT index', 'LRD index', 'Title', 'GT Dimensions ID',
    'LRD Identification code', 'GT Funder', 'LRD Funder', 'GT Total (USD)', 'LRD Total (USD)',
    'GT Project start date', 'LRD Project start date',
])  # explicit columns so a zero-match run still has the right shape (not 0 columns)
print(f"{len(title_matches_df)} (GT, LRD) pairs found")
title_matches_df

13 (GT, LRD) pairs found


,gt_row_id,lrd_row_id,GT index,LRD index,Title,GT Dimensions ID,LRD Identification code,GT Funder,LRD Funder,GT Total (USD),LRD Total (USD),GT Project start date,LRD Project start date
0,1262,1120,7,1120,SusKelpFood – Sustainable ingredients from cul...,grant.9965162,NaN,The Research Council of Norway,Research Council of Norway,2723733.0,NaN,NaT,NaT
1,1731,697,237,697,Impact analysis of the novel food knowledge unit,NaN,NaN,SA Tallinna Teaduspark Tehnopol,Tallinn Technology Park (Tehnopol),26532.0,26532.0,2024-01-23,2024-01-23
2,1732,698,238,698,Microbial proteins as food product?,NaN,NaN,Federal Ministry of Food and Agriculture,Federal Ministry of Food and Agriculture (BMEL),NaN,NaN,2024-06-03,2024-06-03
3,1733,699,239,699,Food4Cells Sustainable cell culture media for ...,NaN,NaN,Research Council of Norway and participating c...,Research Council of Norway,338498.0,338498.0,2025-01-01,2025-01-01
4,1743,708,249,708,Sustainable Proteins from Seaweed,NaN,NaN,NaN,Research Foundation – Flanders (FWO),NaN,NaN,NaT,NaT
5,1743,976,249,976,Sustainable Proteins from Seaweed,NaN,NaN,NaN,Research Foundation – Flanders (FWO),NaN,NaN,NaT,2024-01-10
6,1755,717,259,717,Meat replacement and systems of edibility in A...,NaN,grant.14751793,Research Council of Norway,Research Council of Norway,1127788.0,1127788.0,2025-01-01,2025-01-01
7,1755,1122,259,1122,Meat replacement and systems of edibility in A...,NaN,NaN,Research Council of Norway,Research Council of Norway,1127788.0,NaN,2025-01-01,NaT
8,1855,127,313,127,EAGLE: Enhanced Analytical and Genetics Tools ...,grant.14049012,grant.12930078,Federal Department of Economic Affairs Educati...,UK Research and Innovation (UKRI),283485.0,1048464.0,2025-01-22,2023-01-01
9,1855,288,313,288,EAGLE: Enhanced Analytical and Genetics Tools ...,grant.14049012,NaN,Federal Department of Economic Affairs Educati...,UK Research and Innovation (UKRI),283485.0,274575.0,2025-01-22,2022-01-01


In [44]:
export_for_review(
    title_matches_df, id_cols=['gt_row_id', 'lrd_row_id'],
    out_path=REVIEW_DIR / f'{RUN_TIMESTAMP}_gt_title_match_for_review.csv',
)

Saved 13 candidate matches for review -> data_review\260723_1528_gt_title_match_for_review.csv


,match_key,gt_row_id,lrd_row_id,GT index,LRD index,Title,GT Dimensions ID,LRD Identification code,GT Funder,LRD Funder,GT Total (USD),LRD Total (USD),GT Project start date,LRD Project start date,is_true_match
0,1262__1120,1262,1120,7,1120,SusKelpFood – Sustainable ingredients from cul...,grant.9965162,NaN,The Research Council of Norway,Research Council of Norway,2723733.0,NaN,NaT,NaT,True
1,1731__697,1731,697,237,697,Impact analysis of the novel food knowledge unit,NaN,NaN,SA Tallinna Teaduspark Tehnopol,Tallinn Technology Park (Tehnopol),26532.0,26532.0,2024-01-23,2024-01-23,True
2,1732__698,1732,698,238,698,Microbial proteins as food product?,NaN,NaN,Federal Ministry of Food and Agriculture,Federal Ministry of Food and Agriculture (BMEL),NaN,NaN,2024-06-03,2024-06-03,True
3,1733__699,1733,699,239,699,Food4Cells Sustainable cell culture media for ...,NaN,NaN,Research Council of Norway and participating c...,Research Council of Norway,338498.0,338498.0,2025-01-01,2025-01-01,True
4,1743__708,1743,708,249,708,Sustainable Proteins from Seaweed,NaN,NaN,NaN,Research Foundation – Flanders (FWO),NaN,NaN,NaT,NaT,True
5,1743__976,1743,976,249,976,Sustainable Proteins from Seaweed,NaN,NaN,NaN,Research Foundation – Flanders (FWO),NaN,NaN,NaT,2024-01-10,True
6,1755__717,1755,717,259,717,Meat replacement and systems of edibility in A...,NaN,grant.14751793,Research Council of Norway,Research Council of Norway,1127788.0,1127788.0,2025-01-01,2025-01-01,True
7,1755__1122,1755,1122,259,1122,Meat replacement and systems of edibility in A...,NaN,NaN,Research Council of Norway,Research Council of Norway,1127788.0,NaN,2025-01-01,NaT,True
8,1855__127,1855,127,313,127,EAGLE: Enhanced Analytical and Genetics Tools ...,grant.14049012,grant.12930078,Federal Department of Economic Affairs Educati...,UK Research and Innovation (UKRI),283485.0,1048464.0,2025-01-22,2023-01-01,True
9,1855__288,1855,288,313,288,EAGLE: Enhanced Analytical and Genetics Tools ...,grant.14049012,NaN,Federal Department of Economic Affairs Educati...,UK Research and Innovation (UKRI),283485.0,274575.0,2025-01-22,2022-01-01,True


**Pause here.** Open the exported CSV in `data/`, review the pre-filled `is_true_match` column (defaults to `True` = confirmed match -> gap-fill + remove), flip any false positives to `False` (kept, falls through to the fuzzy-match step below), save it as `..._reviewed.csv` in the same folder (change from '..._for_review'). Then continue.

Guidance: any grants that have a different start data, funding amount, dimensions ID, or other unique distinguishing feature should be switched to FALSE to prevent deduplication and removal.

In [45]:
last_report_data_pre_title = last_report_data_edited.copy()

confirmed_title, rejected_title = apply_reviewed_decisions(
    title_matches_df, REVIEW_DIR / f'{RUN_TIMESTAMP}_gt_title_match_reviewed.csv',
    id_cols=['gt_row_id', 'lrd_row_id'],
)
print(f"{len(confirmed_title)} confirmed, {len(rejected_title)} rejected")

title_changed_indices = set()
title_cells_filled = 0

for _, match in confirmed_title.iterrows():
    gt_idx = int(match['GT index'])
    lrd_idx = int(match['LRD index'])
    gt_row = grants_tracker_data_edited.loc[gt_idx]
    title_cells_filled += gap_fill(gt_row, last_report_data_edited, lrd_idx, gt_lrd_col_map, funding_cols_lrd2, title_changed_indices)

confirmed_gt_row_ids = set(confirmed_title['gt_row_id'])
grants_tracker_data_edited = grants_tracker_data_edited[
    ~grants_tracker_data_edited['gt_row_id'].isin(confirmed_gt_row_ids)
].reset_index(drop=True)

print(f"Filled {title_cells_filled} missing values across {len(title_changed_indices)} rows")
print(f"Removed {len(confirmed_gt_row_ids)} confirmed matches from grants_tracker_data_edited ({len(grants_tracker_data_edited)} remaining)")

11 confirmed, 2 rejected
Filled 21 missing values across 6 rows
Removed 9 confirmed matches from grants_tracker_data_edited (354 remaining)


In [46]:
export_highlighted_diff(last_report_data_pre_title, last_report_data_edited, title_changed_indices,
                         AUDIT_DIR / f'{RUN_TIMESTAMP}_gt_title-match_changes_last_report_data.xlsx')

,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,Minority serving institution,Date added,Last modified,GFI LOS,Link to LOS,GFI partner,Tier,Year project started,End date,lrd_row_id
553,Combinatorial solidification approaches for th...,NaN,NaN,airtable,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,2024-10-31,2024-10-31,NaN,NaN,NaN,NaN,2023,<NA>,553
976,Sustainable Proteins from Seaweed,Because of the generally high standard of livi...,NaN,FRIS,NaN,NaN,EUR,NaN,NaN,NaN,...,NaN,2025-04-14,2025-04-14,NaN,NaN,NaN,NaN,2024,<NA>,976
1001,Occurrence and behavior of spore-forming bacte...,NaN,Vorkommen und Verhalten sporenbildender Bakter...,FEI,524719,524719,EUR,NaN,NaN,524719.00,...,NaN,2025-10-06,2026-03-17,NaN,NaN,NaN,NaN,2023,<NA>,1001
1104,Continuous fermentation of milk protein (Conti...,"Over the past decade, there has been an increa...",NaN,groenprojektbank.dk/,5682434,2557095.3,DKK,969452.0,NaN,738716.42,...,NaN,2025-12-17,2025-12-17,NaN,NaN,NaN,NaN,2024,<NA>,1104
1120,SusKelpFood – Sustainable ingredients from cul...,There is a growing need to produce more food i...,NaN,https://prosjektbanken.forskningsradet.no/,26800000,26800000,NOK,2723733.0,2723733.0,2331600.00,...,NaN,2025-01-14,2026-03-17,NaN,NaN,NaN,NaN,2021,<NA>,1120
1122,Meat replacement and systems of edibility in A...,Alternative proteins can replace meat. Given t...,NaN,https://prosjektbanken.forskningsradet.no/,12000000,12000000,NOK,1127788.0,NaN,1044000.00,...,NaN,2025-04-28,2025-04-28,NaN,NaN,NaN,NaN,2025,<NA>,1122


#### c. Partial title-based match - REVIEW POINT 2 (fuzzy title)

Uses `rapidfuzz.token_sort_ratio`: tokens are sorted before comparing, so word-order differences score highly. Gap-filling is intentionally skipped for partial matches - not reliable enough to fill data across - so a confirmed match here just removes the row (it's a duplicate, already tracked); a rejected one is kept and later appended as new data.

In [47]:
try:
    from rapidfuzz import fuzz as _rfuzz
except ImportError:
    raise ImportError("rapidfuzz not found - run: conda install -c conda-forge rapidfuzz")

lrd_title_list = [
    (idx, normalize_title(val))
    for idx, val in last_report_data_edited['Title'].items()
    if not is_empty(val)
]

partial_match_rows = []
for gt_idx, gt_row in grants_tracker_data_edited.iterrows():
    gt_norm = normalize_title(gt_row.get('EXT_Title'))
    if not gt_norm:
        continue
    for lrd_idx, lrd_norm in lrd_title_list:
        score = _rfuzz.token_sort_ratio(gt_norm, lrd_norm)
        if score >= FUZZY_THRESHOLD:
            partial_match_rows.append({
                'gt_row_id':               gt_row['gt_row_id'],
                'lrd_row_id':              last_report_data_edited.at[lrd_idx, 'lrd_row_id'],
                'GT index':                gt_idx,
                'LRD index':               lrd_idx,
                'Similarity':              score,
                'GT Title':                gt_row.get('EXT_Title'),
                'LRD Title':               last_report_data_edited.at[lrd_idx, 'Title'],
                'GT Dimensions ID':        gt_row.get('Dimensions.ai grant ID'),
                'LRD Identification code': last_report_data_edited.at[lrd_idx, 'Identification code'],
                'GT Total (USD)':          gt_row.get('EXT_Total amount (USD)'),
                'LRD Total (USD)':         last_report_data_edited.at[lrd_idx, 'Total amount (USD)'],
                'GT Project start date':   gt_row.get('EXT_Project start date (estimated)'),
                'LRD Project start date':  last_report_data_edited.at[lrd_idx, 'Project start date'],
            })

partial_matches_df = pd.DataFrame(partial_match_rows, columns=[
    'gt_row_id', 'lrd_row_id', 'GT index', 'LRD index', 'Similarity', 'GT Title', 'LRD Title',
    'GT Dimensions ID', 'LRD Identification code', 'GT Total (USD)', 'LRD Total (USD)',
    'GT Project start date', 'LRD Project start date',
]).sort_values('Similarity', ascending=False).reset_index(drop=True)  # explicit columns so a zero-match run still has the right shape
print(f"{len(partial_matches_df)} (GT, LRD) pairs found at threshold {FUZZY_THRESHOLD}")
partial_matches_df

74 (GT, LRD) pairs found at threshold 85


,gt_row_id,lrd_row_id,GT index,LRD index,Similarity,GT Title,LRD Title,GT Dimensions ID,LRD Identification code,GT Total (USD),LRD Total (USD),GT Project start date,LRD Project start date
0,1729,962,234,962,100.000000,Ecological critique and civic experiments in p...,Ecological critique and civic experiments in p...,NaN,3164-00027A,314570.0,314569.51,2024-02-01,2024-02-01
1,1729,695,234,695,100.000000,Ecological critique and civic experiments in p...,Ecological critique and civic experiments in p...,NaN,NaN,314570.0,314570.00,2024-02-01,2024-02-01
2,1735,700,237,700,100.000000,MadeSweetly - precision fermentation process f...,MadeSweetly - precision fermentation process f...,NaN,NaN,64968.0,64968.00,2024-05-31,2024-05-31
3,1734,822,236,822,100.000000,Socio-ecological research to understand the po...,Socio-ecological research to understand the po...,NaN,grant.14682509,0.0,0.00,2024-09-30,2024-09-30
4,1730,696,235,696,100.000000,Revealing factors determining quality and func...,Revealing factors determining quality and func...,NaN,NaN,305004.0,305004.00,2024-09-11,2024-09-11
...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,1534,680,158,680,93.023256,Engineering safe faba beans by targeting the a...,Engineering safe faba beans by targeting the a...,grant.13984174,"grant.13984174, grant.13909180",232244.0,232655.00,NaT,2024-11-01
70,1779,671,264,671,90.657439,Harnessing the immense potential of\n precisi...,Harnessing the immense potential of precision ...,grant.14917504,grant.14613568,2638230.0,2690896.00,2025-01-01,2025-01-01
71,1857,882,309,882,90.000000,Plant breeding research P3 joint project: 'Rap...,Plant Breeding Research P2 joint project: 'Rap...,grant.13246253,grant.9063165,48759.0,378372.00,2025-10-20,2020-03-01
72,1387,522,63,522,87.619048,Ecologically sustainable food for the elderly ...,Ecologically sustainable food for obese elderly,grant.12908252,grant.9549928,0.0,0.00,2022-09-01,2022-02-01


In [48]:
export_for_review(
    partial_matches_df, id_cols=['gt_row_id', 'lrd_row_id'],
    out_path=REVIEW_DIR / f'{RUN_TIMESTAMP}_gt_partial_title_match_for_review.csv',
)

Saved 74 candidate matches for review -> data_review\260723_1528_gt_partial_title_match_for_review.csv


,match_key,gt_row_id,lrd_row_id,GT index,LRD index,Similarity,GT Title,LRD Title,GT Dimensions ID,LRD Identification code,GT Total (USD),LRD Total (USD),GT Project start date,LRD Project start date,is_true_match
0,1729__962,1729,962,234,962,100.000000,Ecological critique and civic experiments in p...,Ecological critique and civic experiments in p...,NaN,3164-00027A,314570.0,314569.51,2024-02-01,2024-02-01,True
1,1729__695,1729,695,234,695,100.000000,Ecological critique and civic experiments in p...,Ecological critique and civic experiments in p...,NaN,NaN,314570.0,314570.00,2024-02-01,2024-02-01,True
2,1735__700,1735,700,237,700,100.000000,MadeSweetly - precision fermentation process f...,MadeSweetly - precision fermentation process f...,NaN,NaN,64968.0,64968.00,2024-05-31,2024-05-31,True
3,1734__822,1734,822,236,822,100.000000,Socio-ecological research to understand the po...,Socio-ecological research to understand the po...,NaN,grant.14682509,0.0,0.00,2024-09-30,2024-09-30,True
4,1730__696,1730,696,235,696,100.000000,Revealing factors determining quality and func...,Revealing factors determining quality and func...,NaN,NaN,305004.0,305004.00,2024-09-11,2024-09-11,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,1534__680,1534,680,158,680,93.023256,Engineering safe faba beans by targeting the a...,Engineering safe faba beans by targeting the a...,grant.13984174,"grant.13984174, grant.13909180",232244.0,232655.00,NaT,2024-11-01,True
70,1779__671,1779,671,264,671,90.657439,Harnessing the immense potential of\n precisi...,Harnessing the immense potential of precision ...,grant.14917504,grant.14613568,2638230.0,2690896.00,2025-01-01,2025-01-01,True
71,1857__882,1857,882,309,882,90.000000,Plant breeding research P3 joint project: 'Rap...,Plant Breeding Research P2 joint project: 'Rap...,grant.13246253,grant.9063165,48759.0,378372.00,2025-10-20,2020-03-01,True
72,1387__522,1387,522,63,522,87.619048,Ecologically sustainable food for the elderly ...,Ecologically sustainable food for obese elderly,grant.12908252,grant.9549928,0.0,0.00,2022-09-01,2022-02-01,True


**Pause here.** Open the exported CSV in `data/`, review the pre-filled `is_true_match` column (defaults to `True` = confirmed match -> gap-fill + remove), flip any false positives to `False` (kept), save it as `..._reviewed.csv` in the same folder (change from '..._for_review'). Then continue.

Guidance: any grants that have a different start data, funding amount, dimensions ID, or other unique distinguishing feature should be switched to FALSE to prevent deduplication and removal.

In [49]:
confirmed_partial, rejected_partial = apply_reviewed_decisions(
    partial_matches_df, REVIEW_DIR / f'{RUN_TIMESTAMP}_gt_partial_title_match_reviewed.csv',
    id_cols=['gt_row_id', 'lrd_row_id'],
)
print(f"{len(confirmed_partial)} confirmed duplicates (removed, no gap-fill), {len(rejected_partial)} rejected (kept as new)")

confirmed_gt_row_ids = set(confirmed_partial['gt_row_id'])
grants_tracker_data_edited = grants_tracker_data_edited[
    ~grants_tracker_data_edited['gt_row_id'].isin(confirmed_gt_row_ids)
].reset_index(drop=True)
print(f"Removed {len(confirmed_gt_row_ids)} rows from grants_tracker_data_edited ({len(grants_tracker_data_edited)} remaining)")

69 confirmed duplicates (removed, no gap-fill), 5 rejected (kept as new)
Removed 30 rows from grants_tracker_data_edited (324 remaining)


#### d. Append remaining grants tracker rows to the last report's data

Whatever's left in `grants_tracker_data_edited` after all the matching above is genuinely new - append it.

In [50]:
append_extra_map = {
    'Dimensions.ai grant ID': 'Identification code',
}
full_append_map = {**gt_lrd_col_map, **append_extra_map}

gt_for_append = grants_tracker_data_edited.rename(columns={
    gt_col: lrd_col for gt_col, lrd_col in full_append_map.items()
})
gt_for_append = gt_for_append[[c for c in gt_for_append.columns if c in last_report_data_edited.columns]].copy()
gt_for_append['Database'] = 'airtable'

last_report_data_edited = pd.concat([last_report_data_edited, gt_for_append], ignore_index=True)

print(f"Appended {len(grants_tracker_data_edited)} rows from grants_tracker_data_edited")
print(f"Total rows in last_report_data_edited: {len(last_report_data_edited)}")

Appended 324 rows from grants_tracker_data_edited
Total rows in last_report_data_edited: 1504


### Dimensions title matching
#### a. Full title match - REVIEW POINT 3 (exact title)

Matches on `Title translated` in `dimensions_data` against `Title` in `last_report_data_edited` (now including the grants-tracker rows appended above).

In [51]:
lrd_dim_title_map = {}
for idx, val in last_report_data_edited['Title'].items():
    norm = normalize_title(val)
    if norm:
        lrd_dim_title_map.setdefault(norm, []).append(idx)

dim_title_match_rows = []
for dim_idx, dim_row in dimensions_data.iterrows():
    norm = normalize_title(dim_row.get('Title translated'))
    if norm and norm in lrd_dim_title_map:
        for lrd_idx in lrd_dim_title_map[norm]:
            dim_title_match_rows.append({
                'Grant ID':                dim_row['Grant ID'],
                'lrd_row_id':              last_report_data_edited.at[lrd_idx, 'lrd_row_id'],
                'Dim index':               dim_idx,
                'LRD index':               lrd_idx,
                'Title':                   dim_row.get('Title translated'),
                'LRD Identification code': last_report_data_edited.at[lrd_idx, 'Identification code'],
                'Dim Funder':              dim_row.get('Funder'),
                'LRD Funder':              last_report_data_edited.at[lrd_idx, 'Funder name'],
                'Dim Total (USD)':         dim_row.get('Total amount (USD)'),
                'LRD Total (USD)':         last_report_data_edited.at[lrd_idx, 'Total amount (USD)'],
                'Dim Start date':          dim_row.get('Start date'),
                'LRD Project start date':  last_report_data_edited.at[lrd_idx, 'Project start date'],
            })

dim_title_matches_df = pd.DataFrame(dim_title_match_rows, columns=[
    'Grant ID', 'lrd_row_id', 'Dim index', 'LRD index', 'Title', 'LRD Identification code',
    'Dim Funder', 'LRD Funder', 'Dim Total (USD)', 'LRD Total (USD)', 'Dim Start date', 'LRD Project start date',
])  # explicit columns so a zero-match run still has the right shape (not 0 columns)
print(f"{len(dim_title_matches_df)} (Dim, LRD) pairs found")
dim_title_matches_df

0 (Dim, LRD) pairs found


,Grant ID,lrd_row_id,Dim index,LRD index,Title,LRD Identification code,Dim Funder,LRD Funder,Dim Total (USD),LRD Total (USD),Dim Start date,LRD Project start date


In [52]:
export_for_review(
    dim_title_matches_df, id_cols=['Grant ID', 'lrd_row_id'],
    out_path=REVIEW_DIR / f'{RUN_TIMESTAMP}_dim_title_match_for_review.csv',
)

No candidate matches to review - skipping export.


,Grant ID,lrd_row_id,Dim index,LRD index,Title,LRD Identification code,Dim Funder,LRD Funder,Dim Total (USD),LRD Total (USD),Dim Start date,LRD Project start date


**Pause here.** Review the exported CSV - default `is_true_match=True` means "confirmed match, gap-fill + remove". Save as `..._reviewed_{date}.csv`, then continue.

In [53]:
last_report_data_pre_dim_title = last_report_data_edited.copy()

confirmed_dim_title, rejected_dim_title = apply_reviewed_decisions(
    dim_title_matches_df, REVIEW_DIR / f'{RUN_TIMESTAMP}_dim_title_match_reviewed.csv',
    id_cols=['Grant ID', 'lrd_row_id'],
)
print(f"{len(confirmed_dim_title)} confirmed, {len(rejected_dim_title)} rejected")

dim_title_changed_indices = set()
dim_title_cells_filled = 0

for _, match in confirmed_dim_title.iterrows():
    dim_idx = int(match['Dim index'])
    lrd_idx = int(match['LRD index'])
    dim_row = dimensions_data.loc[dim_idx]
    dim_title_cells_filled += gap_fill(dim_row, last_report_data_edited, lrd_idx, col_map, funding_cols_lrd, dim_title_changed_indices)

confirmed_grant_ids = set(confirmed_dim_title['Grant ID'].astype(str))
dimensions_data = dimensions_data[
    ~dimensions_data['Grant ID'].astype(str).isin(confirmed_grant_ids)
].reset_index(drop=True)

print(f"Filled {dim_title_cells_filled} missing values across {len(dim_title_changed_indices)} rows")
print(f"Removed {len(confirmed_grant_ids)} confirmed matches from dimensions_data ({len(dimensions_data)} remaining)")

No candidate matches were exported for review - skipping reviewed-file read.
0 confirmed, 0 rejected
Filled 0 missing values across 0 rows
Removed 0 confirmed matches from dimensions_data (32 remaining)


In [54]:
export_highlighted_diff(last_report_data_pre_dim_title, last_report_data_edited, dim_title_changed_indices,
                         AUDIT_DIR / f'{RUN_TIMESTAMP}_dim_title_changes_last_report_data.xlsx')

,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,Minority serving institution,Date added,Last modified,GFI LOS,Link to LOS,GFI partner,Tier,Year project started,End date,lrd_row_id


#### b. Partial title match - REVIEW POINT 4 (fuzzy title)

The original prototype found far fewer partial matches here (6) than a prior manual pass reportedly found (73). The diagnostic cell below narrows down *why* - it does not resolve the discrepancy on its own; getting the original methodology to compare against is still needed for a definitive answer.

In [55]:
print(f"Candidate pool sizes at this point: dimensions_data={len(dimensions_data)}, last_report_data_edited={len(last_report_data_edited)}")

_lrd_titles = [normalize_title(v) for v in last_report_data_edited['Title'] if not is_empty(v)]
_dim_titles_translated = [normalize_title(v) for v in dimensions_data['Title translated'] if not is_empty(v)]
print(f"Populated 'Title translated' in dimensions_data: {len(_dim_titles_translated)}/{len(dimensions_data)}")

print("\nMatch counts by threshold (token_sort_ratio):")
for threshold in (85, 80, 75, 70):
    count = sum(
        1
        for dim_norm in _dim_titles_translated
        for lrd_norm in _lrd_titles
        if _rfuzz.token_sort_ratio(dim_norm, lrd_norm) >= threshold
    )
    print(f"  threshold {threshold}: {count} (Dim, LRD) pairs")

print(f"\nMatch counts by scorer (threshold={FUZZY_THRESHOLD}):")
for scorer_name, scorer in (('token_sort_ratio', _rfuzz.token_sort_ratio),
                             ('partial_ratio', _rfuzz.partial_ratio),
                             ('WRatio', _rfuzz.WRatio)):
    count = sum(
        1
        for dim_norm in _dim_titles_translated
        for lrd_norm in _lrd_titles
        if scorer(dim_norm, lrd_norm) >= FUZZY_THRESHOLD
    )
    print(f"  {scorer_name}: {count} (Dim, LRD) pairs")

Candidate pool sizes at this point: dimensions_data=32, last_report_data_edited=1504
Populated 'Title translated' in dimensions_data: 32/32

Match counts by threshold (token_sort_ratio):
  threshold 85: 0 (Dim, LRD) pairs
  threshold 80: 0 (Dim, LRD) pairs
  threshold 75: 0 (Dim, LRD) pairs
  threshold 70: 0 (Dim, LRD) pairs

Match counts by scorer (threshold=85):
  token_sort_ratio: 0 (Dim, LRD) pairs
  partial_ratio: 2 (Dim, LRD) pairs
  WRatio: 13739 (Dim, LRD) pairs


In [56]:
lrd_dim_partial_list = [
    (idx, normalize_title(val))
    for idx, val in last_report_data_edited['Title'].items()
    if not is_empty(val)
]

dim_partial_match_rows = []
for dim_idx, dim_row in dimensions_data.iterrows():
    dim_norm = normalize_title(dim_row.get('Title translated'))
    if not dim_norm:
        continue
    for lrd_idx, lrd_norm in lrd_dim_partial_list:
        score = _rfuzz.token_sort_ratio(dim_norm, lrd_norm)
        if score >= FUZZY_THRESHOLD:
            dim_partial_match_rows.append({
                'Grant ID':                dim_row['Grant ID'],
                'lrd_row_id':              last_report_data_edited.at[lrd_idx, 'lrd_row_id'],
                'Dim index':               dim_idx,
                'LRD index':               lrd_idx,
                'Similarity':              score,
                'Dim Title':               dim_row.get('Title translated'),
                'LRD Title':               last_report_data_edited.at[lrd_idx, 'Title'],
                'LRD Identification code': last_report_data_edited.at[lrd_idx, 'Identification code'],
                'Dim Total (USD)':         dim_row.get('Total amount (USD)'),
                'LRD Total (USD)':         last_report_data_edited.at[lrd_idx, 'Total amount (USD)'],
                'Dim Start date':          dim_row.get('Start date'),
                'LRD Project start date':  last_report_data_edited.at[lrd_idx, 'Project start date'],
            })

dim_partial_matches_df = pd.DataFrame(dim_partial_match_rows, columns=[
    'Grant ID', 'lrd_row_id', 'Dim index', 'LRD index', 'Similarity', 'Dim Title', 'LRD Title',
    'LRD Identification code', 'Dim Total (USD)', 'LRD Total (USD)',
    'Dim Start date', 'LRD Project start date',
]).sort_values('Similarity', ascending=False).reset_index(drop=True)  # explicit columns so a zero-match run still has the right shape
print(f"{len(dim_partial_matches_df)} (Dim, LRD) pairs found at threshold {FUZZY_THRESHOLD}")
dim_partial_matches_df

0 (Dim, LRD) pairs found at threshold 85


,Grant ID,lrd_row_id,Dim index,LRD index,Similarity,Dim Title,LRD Title,LRD Identification code,Dim Total (USD),LRD Total (USD),Dim Start date,LRD Project start date


In [57]:
export_for_review(
    dim_partial_matches_df, id_cols=['Grant ID', 'lrd_row_id'],
    out_path=REVIEW_DIR / f'{RUN_TIMESTAMP}_dim_partial_title_match_for_review.csv',
)

No candidate matches to review - skipping export.


,Grant ID,lrd_row_id,Dim index,LRD index,Similarity,Dim Title,LRD Title,LRD Identification code,Dim Total (USD),LRD Total (USD),Dim Start date,LRD Project start date


**Pause here.** Review the exported CSV - default `is_true_match=True` means "confirmed duplicate, remove" (no gap-fill). Save as `..._reviewed_{date}.csv`, then continue.

In [58]:
confirmed_dim_partial, rejected_dim_partial = apply_reviewed_decisions(
    dim_partial_matches_df, REVIEW_DIR / f'{RUN_TIMESTAMP}_dim_partial_title_match_reviewed.csv',
    id_cols=['Grant ID', 'lrd_row_id'],
)
print(f"{len(confirmed_dim_partial)} confirmed duplicates (removed, no gap-fill), {len(rejected_dim_partial)} rejected (kept as new)")

confirmed_grant_ids = set(confirmed_dim_partial['Grant ID'].astype(str))
dimensions_data = dimensions_data[
    ~dimensions_data['Grant ID'].astype(str).isin(confirmed_grant_ids)
].reset_index(drop=True)
print(f"Removed {len(confirmed_grant_ids)} rows from dimensions_data ({len(dimensions_data)} remaining)")

No candidate matches were exported for review - skipping reviewed-file read.
0 confirmed duplicates (removed, no gap-fill), 0 rejected (kept as new)
Removed 0 rows from dimensions_data (32 remaining)


### Final outputs

Two separate accumulating tables, deliberately kept apart because they have different schemas and different lifecycles:

- **`{DEDUP_TABLE}`** (DuckDB table, per-run) - the genuinely new/untracked grants, DSL/Dimensions schema. This is the programmatic contract S3 reads from (it's an automated script, not a notebook, so it needs a stable table to read, not an Excel file someone has to re-save). S3 then grows the permanent `funding_classified` table from these rows.
- **`funding_curated`** (DuckDB table, rebuilt fresh every run) - the gap-filled, merged last-report + Grants Tracker dataset, in the GFI tracker schema (~80 columns, same shape as `Funding{LAST_REPORT_YEAR}_inscope.xlsx`). Written via `CREATE OR REPLACE`, not accumulated - it's fully recomputed from whichever `LAST_REPORT_FILE` / `GRANTS_TRACKER_FILE` are configured each time this notebook runs, so next cycle you can simply point `LAST_REPORT_FILE` at this run's `funding_curated` export and start again from there.

Note for later: a future stage (not built yet, same boundary as the deferred manual-review/category-labelling work) will map `funding_classified` rows that clear LLM scope screening *and* human review into `funding_curated`'s schema and append them there - `funding_curated` is meant to keep growing that way over time, on top of what this notebook seeds it with.

In [59]:
con = duckdb.connect(str(DB_PATH))
con.execute(f"CREATE OR REPLACE TABLE {DEDUP_TABLE} AS SELECT * FROM dimensions_data")

last_report_data_edited['date_curated'] = RUN_DATE
con.execute("CREATE OR REPLACE TABLE funding_curated AS SELECT * FROM last_report_data_edited")
con.close()

print(f"Saved {len(dimensions_data)} new/untracked grants -> table '{DEDUP_TABLE}' (S3's input)")
print(f"Saved {len(last_report_data_edited)} in-scope rows -> table 'funding_curated'")

# Human-readable Excel mirror as well
last_report_data_edited.to_excel(AUDIT_DIR / f'{RUN_TIMESTAMP}_last_report_data_edited_{RUN_TABLE}.xlsx', index=False)
print(f"Excel copy also saved to {AUDIT_DIR}/last_report_data_edited_{RUN_TABLE}.xlsx")

Saved 32 new/untracked grants -> table 'run_260723_1526_dedup' (S3's input)
Saved 1504 in-scope rows -> table 'funding_curated'
Excel copy also saved to data_audit/last_report_data_edited_run_260723_1526.xlsx
